In [1]:
import pybamm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
import dfols
import signal
from tqdm import tqdm
from scipy.integrate import solve_ivp
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import savgol_filter
from scipy.signal import find_peaks
from scipy import interpolate, integrate
from stopit import threading_timeoutable as timeoutable
import os, sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
from batfuns import *
plt.rcParams = set_rc_params(plt.rcParams)
import winsound
from pybamm import exp, constants, Parameter
import pickle
import matplotlib as mpl
pd.options.mode.chained_assignment = None


eSOH_DIR = "../data/esoh_R/"
oCV_DIR = "../data/ocv/"
cyc_DIR = "../data/cycling/"
fig_DIR = "../figures/figures_p2d/"
res_DIR = "../data/results_p2d/"
resistance_DIR = "../data/resistance/"
%matplotlib widget

In [2]:
cell = 1

In [3]:
cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)

In [4]:
sno = 0

# DFN

In [5]:
N

array([  0,  18,  57,  93, 134, 175, 216, 257, 298, 339], dtype=int64)

In [6]:
dfn = pybamm.lithium_ion.DFN(
    {
        # "SEI": "ec reaction limited",
        # "loss of active material": "stress-driven",
        # "lithium plating": "irreversible",
        "stress-induced diffusion": "false",
        "particle mechanics":"swelling only",
    }
)
param=dfn.param
parameter_values = get_parameter_values()   
# sim_des = sim_des+'_cv'
cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
Ns = np.insert(N_0[1:]-1,0,0)
eps_n_data,eps_p_data,c_rate_c,c_rate_d,dis_set,Temp,SOC_0 = init_exp(cell_no,dfe,dfn,parameter_values)
pybamm.set_logging_level("WARNING")
par_val = {}
# Room temp
par_val[0] = [4.0312e-08,1.8157e-07,1.0776,2.3586e-09,-4.9170e-09,-1.4406e-09,4.60788219e-16,4.56607447e-19]
parameter_values = get_parameter_values()
parameter_values.update(
    {   
        "Positive electrode diffusion coefficient activation energy [J.mol-1]": 0,
        "Negative electrode diffusion coefficient activation energy [J.mol-1]": 0,
        "Positive electrode reference exchange-current density activation energy [J.mol-1]": 0,
        "Negative electrode reference exchange-current density activation energy [J.mol-1]": 0,
        "Positive electrode diffusion coefficient [m2.s-1]": 8e-15,
        "Negative electrode diffusion coefficient [m2.s-1]": 8e-14,
        "Positive electrode reference exchange-current density [A.m-2(m3.mol)1.5]": 3.377e-06,
        "Negative electrode reference exchange-current density [A.m-2(m3.mol)1.5]":	3.183e-06,
        "Negative electrode active material volume fraction": eps_n_data,
        "Positive electrode active material volume fraction": eps_p_data,
        "Initial temperature [K]": 273.15+Temp,
        "Ambient temperature [K]": 273.15+Temp,
        "Positive electrode LAM constant proportional term [s-1]": par_val[sno][0],
        "Negative electrode LAM constant proportional term [s-1]": par_val[sno][1],
        "Positive electrode LAM constant proportional term 2 [s-1]": par_val[sno][5],
        "Negative electrode LAM constant proportional term 2 [s-1]": par_val[sno][4],
        "Positive electrode LAM constant exponential term": par_val[sno][2],
        "Negative electrode LAM constant exponential term": par_val[sno][2],
        "SEI kinetic rate constant [m.s-1]":  par_val[sno][6], #1.08494281e-16 , 
        "EC diffusivity [m2.s-1]": par_val[sno][7],#8.30909086e-19,
        "SEI growth activation energy [J.mol-1]": 1.87422275e+04,#1.58777981e+04,
        "Lithium plating kinetic rate constant [m.s-1]": par_val[sno][3],
        "Initial inner SEI thickness [m]": 0e-09,
        "Initial outer SEI thickness [m]": 5e-09,
        "Li plating resistivity [Ohm.m]": 30000,
        "SEI resistivity [Ohm.m]": 30000.0,
        "Negative electrode partial molar volume [m3.mol-1]": 7e-06,
        "Negative electrode LAM min stress [Pa]": 0,
        "Negative electrode LAM max stress [Pa]": 0,
        "Positive electrode LAM min stress [Pa]": 0,
        "Positive electrode LAM max stress [Pa]": 0,
        # "Negative electrode diffusion coefficient [m2.s-1]": 8e-14,
        # "Positive electrode diffusion coefficient [m2.s-1]": 8e-15,
        # "Negative electrode critical stress [Pa]": 20e+06,
        # "Positive electrode critical stress [Pa]": 40e+06,
    },
    check_already_exists=False,
)
if cell == 13 or cell == 16:
    parameter_values.update(
        {
            "Negative electrode partial molar volume [m3.mol-1]":	0.747*7e-06,
        },
        check_already_exists=False,
    )

experiment = pybamm.Experiment(
    [
        ("Rest for 10 min",
        "Charge at "+c_rate_c+" until 4.2V", 
        "Hold at 4.2V until C/100",
        "Discharge at "+c_rate_d+dis_set,)
    ],
    termination="50% capacity",
)
SOC_0 = 0
sols = []
parameter_values.update(
    {
        "Negative electrode LAM min stress [Pa]": 0,
        "Negative electrode LAM max stress [Pa]": 0,
        "Positive electrode LAM min stress [Pa]": 0,
        "Positive electrode LAM max stress [Pa]": 0,

    },
    check_already_exists=False,
)
esoh_model = pybamm.lithium_ion.ElectrodeSOH()
esoh_sim = pybamm.Simulation(esoh_model, parameter_values=parameter_values)
c_n_max = parameter_values.evaluate(param.n.prim.c_max)
c_p_max = parameter_values.evaluate(param.p.prim.c_max)
Cn = parameter_values.evaluate(param.n.cap_init)
Cp = parameter_values.evaluate(param.p.cap_init)
n_Li_init = parameter_values.evaluate(param.n_Li_particles_init)
n_Li = n_Li_init

Vmin = 3.0
Vmax = 4.2
esoh_sol = esoh_sim.solve(
    [0],
    inputs={"V_min": Vmin, "V_max": Vmax, "C_n": Cn, "C_p": Cp, "n_Li": n_Li_init},
    solver=pybamm.AlgebraicSolver(),
)

parameter_values.update(
    {
        "Initial concentration in negative electrode [mol.m-3]": esoh_sol[
            "x_0"
        ].data[0]
        * c_n_max,
        "Initial concentration in positive electrode [mol.m-3]": esoh_sol[
            "y_0"
        ].data[0]
        * c_p_max,
        
    }
)

In [7]:
R_sei = parameter_values["SEI resistivity [Ohm.m]"]
R_plated_Li = parameter_values["Li plating resistivity [Ohm.m]"]
C_sei_ec = parameter_values["EC initial concentration in electrolyte [mol.m-3]"]
D_sei = parameter_values["EC diffusivity [m2.s-1]"]
k_sei0 = parameter_values["SEI kinetic rate constant [m.s-1]"]
E_seia = parameter_values["SEI growth activation energy [J.mol-1]"]
T_ref  = parameter_values["Reference temperature [K]"]
R = parameter_values.evaluate(param.R)
F = parameter_values.evaluate(param.F)
alp_pl = parameter_values["Lithium plating transfer coefficient"]
alp_sei = 0.5
k_pl0 = parameter_values["Lithium plating kinetic rate constant [m.s-1]"]
U_pl = parameter_values["Li plating open-circuit potential [V]"]
U_sei = parameter_values["SEI open-circuit potential [V]"]
R_p_n = parameter_values["Negative particle radius [m]"]
e_s_n_0 = parameter_values["Negative electrode active material volume fraction"]
a_s_n_0 = 3*parameter_values["Negative electrode active material volume fraction"]/R_p_n
omega_sei = parameter_values["Inner SEI partial molar volume [m3.mol-1]"]
A = parameter_values["Cell cooling surface area [m2]"]
l_n = parameter_values["Negative electrode thickness [m]"]
delta_sei_0 = parameter_values["Initial outer SEI thickness [m]"]
delta_sei = delta_sei_0

In [8]:
n_Li_a = []
x_0_a = []
x_100_a = []
y_0_a = []
y_100_a = []
delta_sei_a = []
time_a = []
Qmax_a = []
Ahth_a = []
Ahth = 0
time =0

x_0_a.append(esoh_sol["x_0"].data[0])
x_100_a.append(esoh_sol["x_100"].data[0])
y_0_a.append(esoh_sol["y_0"].data[0])
y_100_a.append(esoh_sol["y_100"].data[0])
n_Li_a.append(n_Li_init)
delta_sei_a.append(delta_sei_0)
time_a.append(time)

In [9]:
for i in tqdm(range(N[-1])):
    sim_long = pybamm.Simulation(dfn, experiment=experiment, parameter_values=parameter_values, 
                                solver=pybamm.CasadiSolver("safe"))
    sol1 = sim_long.solve(initial_soc=SOC_0)
    model = sim_long.solution.all_models[0]

    t = sol1["Time [s]"].entries
    delta_phi = sol1["X-averaged negative electrode surface potential difference [V]"].entries
    j = sol1[
            "X-averaged negative"
            + " electrode total interfacial current density [A.m-2]"
        ].entries
    T = sol1["X-averaged negative electrode temperature [K]"].entries
    Q = sol1["Discharge capacity [A.h]"].entries
    c_ss_n = sol1["X-averaged negative particle surface concentration [mol.m-3]"].entries
    c_save_n = sol1["R-averaged negative particle concentration [mol.m-3]"].entries
    c_save_n = np.average(c_save_n,axis=0)
    # c_e_n = sol1["Negative electrolyte concentration [mol.m-3]"].entries
    U_p = sol1["X-averaged positive electrode open circuit potential [V]"].entries
    U_n = sol1["X-averaged negative electrode open circuit potential [V]"].entries
    a_s_n = 3*sol1["X-averaged negative electrode active material volume fraction"].entries/R_p_n

    eta_SEI = delta_phi - j * delta_sei * R_sei  - U_sei
    k_SEI = k_sei0*np.exp(-E_seia*(1-T/T_ref)/R/T_ref)
    SEI_exp = k_SEI*np.exp(-alp_sei*F*eta_SEI/R/T)
    j_sei = -C_sei_ec*F/(1/SEI_exp+delta_sei/D_sei)

    c_sei = integrate.cumtrapz(-a_s_n*j_sei/2/F,t)
    c_sei = np.insert(c_sei, 0, 0)

    # sei layer thickness
    delta_sei += c_sei[-1]*omega_sei/a_s_n_0 

    i_side = -A*l_n*a_s_n*(j_sei)/2
    n_li_loss = integrate.cumtrapz(i_side/F,t)
    n_li_loss = np.insert(n_li_loss, 0, 0)
    # moles of lithium
    n_Li-= n_li_loss[-1]
    Qmax = max(abs(Q))
    Ahth += np.cumsum(abs(np.diff(Q)))[-1]
    time += t[-1]

    esoh_sol = esoh_sim.solve(
        [0],
        inputs={"V_min": Vmin, "V_max": Vmax, "C_n": Cn, "C_p": Cp, "n_Li": n_Li},
    )
    esoh_sim.built_model.set_initial_conditions_from(esoh_sol)
    ics = {}
    x_100 = esoh_sol["x_100"].data[0]
    y_100 = esoh_sol["y_100"].data[0]
    x_0 = esoh_sol["x_0"].data[0]
    y_0 = esoh_sol["y_0"].data[0]
    x_0_a.append(x_0)
    x_100_a.append(x_100)
    y_0_a.append(y_0)
    y_100_a.append(y_100)
    n_Li_a.append(n_Li)
    delta_sei_a.append(delta_sei)
    time_a.append(time)
    Qmax_a.append(Qmax)
    Ahth_a.append(Ahth)

    for var in model.initial_conditions:
        if var.name == "Negative particle concentration":
            ics[var.name] = ((x_100-x_0)*SOC_0+x_0) * np.ones((model.variables[var.name].size, 2))
        elif var.name == "Positive particle concentration":
            ics[var.name] = ((y_100-y_0)*SOC_0+y_0)  * np.ones((model.variables[var.name].size, 2))
        if var.name == "Discharge capacity [A.h]":
            ics[var.name] = np.zeros(1)
        elif var.name == "Porosity times concentration":
            for child in var.children:
                value = sol1[child.name].data
                val = np.average(value[:,0])
                ics[child.name] = val * np.ones((model.variables[var.name].size, 1))
        elif var.name == "Negative electrode potential":
            value = sol1[var.name].data
            ics[var.name] = value[:,0]
        elif var.name == "Positive electrode potential":
            value = sol1[var.name].data
            ics[var.name] = value[:,0]
        elif var.name == "Electrolyte potential":
            for child in var.children:
                value = sol1[child.name].data
                val = np.average(value[:,0])
                ics[child.name] = val * np.ones((model.variables[var.name].size, 1))
    model.set_initial_conditions_from(ics)

100%|██████████| 339/339 [40:10<00:00,  7.11s/it]


In [10]:
with open('dfn_sim_sei_c5_eSOH.pickle', 'wb') as handle:
        pickle.dump((x_100_a,y_100_a,x_0_a,y_0_a,n_Li_a,delta_sei_a,Cp,Cn,time_a,Qmax_a,Ahth_a), handle, protocol=pickle.HIGHEST_PROTOCOL)